# Assignment 3 - Regularization + Gradient Descent

## Library imports
- all imports from other libraries are in below cell

In [ ]:
from ucimlrepo import fetch_ucirepo 
import pandas as pd
import numpy as np
from sklearn.linear_model import LassoCV, RidgeCV
# import matplotlib.pyplot as plt
import random

## Dataset `Communities and Crime`
### Importing

In [2]:
  
# fetch dataset 
communities_and_crime = fetch_ucirepo(id=183) 
  
# data (as pandas dataframes) 
X = communities_and_crime.data.features 
y = communities_and_crime.data.targets 
  
# metadata 
print(communities_and_crime.metadata) 
  
# variable information 
print(communities_and_crime.variables) 

print(type(y))
print(y.columns)

{'uci_id': 183, 'name': 'Communities and Crime', 'repository_url': 'https://archive.ics.uci.edu/dataset/183/communities+and+crime', 'data_url': 'https://archive.ics.uci.edu/static/public/183/data.csv', 'abstract': 'Communities within the United States. The data combines socio-economic data from the 1990 US Census, law enforcement data from the 1990 US LEMAS survey, and crime data from the 1995 FBI UCR.', 'area': 'Social Science', 'tasks': ['Regression'], 'characteristics': ['Multivariate'], 'num_instances': 1994, 'num_features': 127, 'feature_types': ['Real'], 'demographics': ['Race', 'Age', 'Income', 'Occupation'], 'target_col': ['ViolentCrimesPerPop'], 'index_col': None, 'has_missing_values': 'yes', 'missing_values_symbol': 'NaN', 'year_of_dataset_creation': 2002, 'last_updated': 'Mon Mar 04 2024', 'dataset_doi': '10.24432/C53W3X', 'creators': ['Michael Redmond'], 'intro_paper': {'ID': 405, 'type': 'NATIVE', 'title': 'A data-driven software tool for enabling cooperative information s

### Pre-processing
- Remove the attributes state, county, community, community name, and fold (columns 1 to 5).

In [3]:
X = X.drop(columns = ['state', 'county', 'community', 'communityname', 'fold'])

- dropping columns with high percentage of missing values

In [4]:
na_percentage = (X.select_dtypes(include=['str', 'object'], ) == '?').mean(axis=0)
# na_percentage = na_percentage[na_percentage > 0.8]
drop_cols = na_percentage[na_percentage > 0.80].index.tolist()

X = X.drop(columns=drop_cols)

# remaining column (`OtherPerCap`) has only 1 row with NA
X = X[~(X['OtherPerCap'] == '?')]

# converting it to float
X['OtherPerCap'] = X['OtherPerCap'].astype('float')


- splitting into train and test set

In [5]:
SPLIT = 0.8

split_index = int(SPLIT * X.shape[0])
train_X = X.iloc[:split_index]
train_y = y.iloc[:split_index]

test_X = X.iloc[split_index:]
test_y = y.iloc[split_index:]

## Regularization
- Fit linear regression using ridge and lasso regression. Try to find the optimal regularization parameter.
- Use the ridge and lasso functions from Scikit-learn. For the selection of the regularization parameter use LassoCV and RidgeCV
- Compare the attributes selected with forward attribute selection (the results from the
previous assignment) with the attributes lasso selected.
- Comment on the results

In [6]:

def ridge_fit(train_X, train_y, alpha = [0.1, 0.25, 0.5, 1, 2, 4, 10]):
    
    model = RidgeCV(alphas = alpha)
    model.fit(train_X, train_y)
    print(f"Best fit (Ridge) with alpha value: {model.alpha_}")
    return model

def lasso_fit(train_X, train_y, alpha = 1):
    model = LassoCV(max_iter=50_000)
    model.fit(train_X, train_y)
    print(f"Best fit (Lasso) with alpha value: {model.alpha_}")
    return model



### Testing Ridge

In [7]:

ridge_model = ridge_fit(train_X, train_y)

test_y_preds = ridge_model.predict(test_X)
ridge_test_mse = np.mean((test_y_preds - test_y.to_numpy()) ** 2)
print(f"Ridge regularized linear model mse value: {ridge_test_mse:.3f}")

Best fit (Ridge) with alpha value: 10.0
Ridge regularized linear model mse value: 0.050


### Testing Lasso

In [8]:

lasso_model = lasso_fit(train_X, train_y, alpha=[1])

test_y_preds = lasso_model.predict(test_X)
lasso_test_mse = np.mean((test_y_preds - test_y.to_numpy()) ** 2)
print(f"Lasso regularized linear model mse value: {lasso_test_mse:.3f}")

d:\SU_ML_assignments\assignment_3\.venv\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:1837: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


Best fit (Lasso) with alpha value: 0.003965895831135896
Lasso regularized linear model mse value: 0.049


In [9]:


lasso_cols = X.columns[lasso_model.coef_ != 0].to_list()
lasso_ommitted_cols = X.columns[lasso_model.coef_ == 0].to_list()
print(f"Columns selected with Lasso regularization: {lasso_cols}")
print(f"Ommited columns: {lasso_ommitted_cols}")
print(lasso_model.coef_)

Columns selected with Lasso regularization: []
Ommited columns: ['population', 'householdsize', 'racepctblack', 'racePctWhite', 'racePctAsian', 'racePctHisp', 'agePct12t21', 'agePct12t29', 'agePct16t24', 'agePct65up', 'numbUrban', 'pctUrban', 'medIncome', 'pctWWage', 'pctWFarmSelf', 'pctWInvInc', 'pctWSocSec', 'pctWPubAsst', 'pctWRetire', 'medFamInc', 'perCapInc', 'whitePerCap', 'blackPerCap', 'indianPerCap', 'AsianPerCap', 'OtherPerCap', 'HispPerCap', 'NumUnderPov', 'PctPopUnderPov', 'PctLess9thGrade', 'PctNotHSGrad', 'PctBSorMore', 'PctUnemployed', 'PctEmploy', 'PctEmplManu', 'PctEmplProfServ', 'PctOccupManu', 'PctOccupMgmtProf', 'MalePctDivorce', 'MalePctNevMarr', 'FemalePctDiv', 'TotalPctDiv', 'PersPerFam', 'PctFam2Par', 'PctKids2Par', 'PctYoungKids2Par', 'PctTeen2Par', 'PctWorkMomYoungKids', 'PctWorkMom', 'NumIlleg', 'PctIlleg', 'NumImmig', 'PctImmigRecent', 'PctImmigRec5', 'PctImmigRec8', 'PctImmigRec10', 'PctRecentImmig', 'PctRecImmig5', 'PctRecImmig8', 'PctRecImmig10', 'PctSpea

- comment on above - I think it has to do with the dataset
- on the wine quality it does not have all coefficients 0
- tried multiple fixes - don't know what to do

## Dataset `Wine Quality`
### Importing

In [10]:
# fetch dataset 
wine_quality = fetch_ucirepo(id=186) 
  
# data (as pandas dataframes) 
X = wine_quality.data.features 
y = wine_quality.data.targets

full_data = wine_quality.data.original
# metadata 
print(wine_quality.metadata) 
  
# variable information 
print(wine_quality.variables) 

{'uci_id': 186, 'name': 'Wine Quality', 'repository_url': 'https://archive.ics.uci.edu/dataset/186/wine+quality', 'data_url': 'https://archive.ics.uci.edu/static/public/186/data.csv', 'abstract': 'Two datasets are included, related to red and white vinho verde wine samples, from the north of Portugal. The goal is to model wine quality based on physicochemical tests (see [Cortez et al., 2009], http://www3.dsi.uminho.pt/pcortez/wine/).', 'area': 'Business', 'tasks': ['Classification', 'Regression'], 'characteristics': ['Multivariate'], 'num_instances': 4898, 'num_features': 11, 'feature_types': ['Real'], 'demographics': [], 'target_col': ['quality'], 'index_col': None, 'has_missing_values': 'no', 'missing_values_symbol': None, 'year_of_dataset_creation': 2009, 'last_updated': 'Wed Nov 15 2023', 'dataset_doi': '10.24432/C56S3T', 'creators': ['Paulo Cortez', 'A. Cerdeira', 'F. Almeida', 'T. Matos', 'J. Reis'], 'intro_paper': {'ID': 252, 'type': 'NATIVE', 'title': 'Modeling wine preferences

### Pre-processing
- Choose only the white wine data
- Pre-process your data.


In [12]:
# filter only white wine
X = X[wine_quality.data.original['color'] == 'white']
y = y[wine_quality.data.original['color'] == 'white']


# * - no missing data
# * - only 19 rows with a 0 value, in 1 same column, but out of 6k rows that's a small num
# todo - come back and re-check if some pre-processing needs to be done


SPLIT = 0.8

split_index = int(SPLIT * X.shape[0])
train_X = X.iloc[:split_index]
train_y = y.iloc[:split_index]

test_X = X.iloc[split_index:]
test_y = y.iloc[split_index:]

# standardize values - using data from training set only
train_X = (train_X - train_X.mean()) / train_X.std()
test_X = (test_X - train_X.mean()) / train_X.std()

## Gradient Descent
- implement from scratch


### (Batch) Gradient Descent

In [13]:
class BatchGD:
    # todo - optional arg batch size - default to size of dataset
    # ? to have like something in-between like a
    # ? mini-batch gradient descent, 
    # ? not just either whole dataset or just 1 sample

    def __init__(self, input_size):
        # input_size == number of columns
        self.coef_ = np.zeros(input_size)
        self.ridge_alpha = 1

    def eval_loss(self, X_input, y_actual):
        X = X_input.to_numpy()
        y = np.asarray(y_actual).ravel()
        mse = np.mean(((X @ self.coef_) - y) ** 2)
        ridge = self.ridge_alpha * np.sum(self.coef_ ** 2)
        return mse + ridge

    def eval_mse(self, X_input, y_actual):
        X = X_input.to_numpy()
        y = np.asarray(y_actual).ravel()
        mse = np.mean(((X @ self.coef_) - y) ** 2)
        return mse
    
    def eval_gradient(self, X_input, y_actual):
        X = X_input.to_numpy()
        y = np.asarray(y_actual).ravel()

        residuals = X @ self.coef_ - y
        gradient = (2/X.shape[0]) * (X.T @ residuals) + self.ridge_alpha * 2 * self.coef_
        return gradient

    def fit(self, input_X, y_actual, step_size, tolerance=1e-3, iter_count = 10_000):
        for i in range(iter_count):
            grad = self.eval_gradient(input_X, y_actual)
            if np.linalg.norm(grad) < tolerance:
                break
            self.coef_ = self.coef_ - step_size * grad
        print(f"Fitting with batch gradient descent ran for {i} epochs, with tolerance {tolerance}!")
        

    def predict(self, input):
        X = input.to_numpy()
        return X @ self.coef_
        

### Stochastic Gradient Descent

In [14]:
class StochGD:
    

    def __init__(self, input_size):
        # input_size == number of columns
        self.coef_ = np.zeros(input_size)
        self.ridge_alpha = 1

    def eval_loss(self, X_input, y_actual):
        X = X_input.to_numpy()
        y = y_actual.to_numpy()
        mse = np.mean(((X @ self.coef_) - y) ** 2)
        ridge = self.ridge_alpha * np.sum(self.coef_ ** 2)
        return mse + ridge

    def eval_mse(self, X_input, y_actual):
        X = X_input.to_numpy()
        y = y_actual.to_numpy()
        mse = np.mean(((X @ self.coef_) - y) ** 2)
        return mse
    
    def eval_gradient(self, X, y):
        gradient = np.zeros(len(self.coef_))

        residual = X @ self.coef_ - y
        gradient = 2 * residual * (X.T) + self.ridge_alpha * 2 * self.coef_
        return gradient

    
    def fit(self, input_X, y_actual, step_size, tolerance=1e-3, iter_count=10_000, seed=42):
        rng = np.random.default_rng(seed)   # created once, so the state advances each epoch

        X = input_X.to_numpy()              # convert once, outside the loop
        y = np.asarray(y_actual).ravel()
        n = X.shape[0]

        prev = self.eval_loss(input_X, y_actual)
        for i in range(iter_count):
            for j in rng.permutation(n):    # new order every epoch
                grad = self.eval_gradient(X[j], y[j])
                self.coef_ = self.coef_ - step_size * grad

            curr = self.eval_loss(input_X, y_actual)
            if abs(prev - curr) < tolerance:
                break
            prev = curr

        print(f"Fitting with stochastic gradient descent ran for {i + 1} epochs, with tolerance {tolerance}!")

    def predict(self, input):
        X = input.to_numpy()
        return X @ self.coef_
        

### Testing
-  on the `Wine quality` data.
- Test different learning rates and try to find a good one
- Comment on the results
- Compare the time to convergence and the results of the two gradient descents you implemented

In [15]:
# get coefficients from RidgeCV
ridge_model = ridge_fit(train_X, train_y)

y_pred = ridge_model.predict(test_X)
sk_test_mse = np.mean((np.asarray(test_y).ravel() - y_pred) ** 2)

Best fit (Ridge) with alpha value: 10.0


In [16]:

bgd = BatchGD(train_X.shape[1])

bgd.fit(train_X, train_y, step_size=0.001)
# default tolerance = 1e-3, max_iters = 10_000

batch_train_mse = bgd.eval_mse(train_X, train_y)
batch_test_mse = bgd.eval_mse(test_X, test_y)

print(f"Batch train MSE loss: {batch_train_mse}")
print(f"Batch test MSE loss: {batch_test_mse}")

Fitting with batch gradient descent ran for 2073 epochs, with tolerance 0.001!
Batch train MSE loss: 35.22240228693272
Batch test MSE loss: 40.19259861813302


In [20]:

sgd = StochGD(train_X.shape[1])

sgd.fit(train_X, train_y, step_size=0.001)
# default tolerance = 1e-3, max_iters = 10_000

stoch_train_mse = sgd.eval_mse(train_X, train_y)
stoch_test_mse = sgd.eval_mse(test_X, test_y)

print(f"Stochastic train MSE loss: {stoch_train_mse}")
print(f"Stochastic test MSE loss: {stoch_test_mse}")

Fitting with stochastic gradient descent ran for 38 epochs, with tolerance 0.001!
Stochastic train MSE loss: 35.5892007104125
Stochastic test MSE loss: 71.81451542215859


In [21]:
print(f"RidgeCV coefficients:")
print(ridge_model.coef_)
print(f"BGD coefficients:")
print(bgd.coef_)
print(f"SGD coefficients:")
print(sgd.coef_)

RidgeCV coefficients:
[ 0.02942246 -0.17561489 -0.00482748  0.38006839 -0.00307931  0.08206094
 -0.01124383 -0.40613742  0.10705838  0.08363171  0.28655953]
BGD coefficients:
[-0.0329754  -0.08229206  0.00402691  0.03040986 -0.05066545  0.04380168
 -0.02897415 -0.08037059  0.0317895   0.03122783  0.1722941 ]
SGD coefficients:
[ 0.20940367  0.02500391  0.02225155 -0.08803334  0.1411678   0.22304181
  0.02548001 -0.10768058 -0.10998853 -0.13974288  0.17822193]


In [22]:
print(f"SK Ridge test MSE: {sk_test_mse}")
print(f"Batch test MSE: {batch_test_mse}")
print(f"Stoch test MSE: {stoch_test_mse}")

SK Ridge test MSE: 54.82307765591105
Batch test MSE: 40.19259861813302
Stoch test MSE: 71.81451542215859


### Thoughts
- takes a lot more for stochastic gradient descent to converge
- for stochastic
    - large step size - won't ever converge, will jump around the local optima
    - small step size - takes too long
    - also tested with tolerance 1e-4, but it had 4x higher MSE on test set, and ran for 20x longer